In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_access_management/table_permissions"
tgt_silver_table = "data_governance.silver_access_management.table_permissions"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:
df = df.selectExpr(
"grantor as grantor",
"grantee as grantee",
"table_catalog as catalog_name",
"table_schema as schema_name",
"table_name as table_name",
"privilege_type as privilege_type",
"is_grantable as is_grantable",
"inherited_from as inherited_from",
"load_timestamp as load_timestamp"
)


df = df.withColumn("grantor", trim(lower("grantor"))) \
       .withColumn("grantee", trim(lower("grantee"))) \
       .withColumn("catalog_name", trim(lower("catalog_name"))) \
       .withColumn("schema_name", trim(lower("schema_name"))) \
       .withColumn("table_name", trim(lower("table_name"))) \
       .withColumn("privilege_type", trim(lower("privilege_type")))

df = df.withColumn("privilege_type", lower(trim(col("privilege_type"))))

df = df.withColumn(
"is_grantable_flag",
when(col("is_grantable") == "YES", True).otherwise(False)
)

df = df.filter(
col("catalog_name").isNotNull() &
col("schema_name").isNotNull() &
col("table_name").isNotNull()
)

df = df.withColumn(
"inherited_from",
when(col("inherited_from") == "NONE", None)
.otherwise(col("inherited_from"))
)

df = df.withColumn(
"full_table_path",
concat_ws(".", "catalog_name", "schema_name", "table_name")
)

df = df.withColumn(
    "permission_category",
    when(col("privilege_type").isin("create", "drop", "alter", "truncate"), "DDL")
    .when(col("privilege_type").isin("insert", "update", "delete"), "DML")
    .when(col("privilege_type").isin("commit", "savepoint", "rollback"), "TCL")
    .when(col("privilege_type").isin("select"), "DQL")
    .when(col("privilege_type").isin("grant", "revoke"), "DCL")
    .otherwise("OTHER")
)


In [0]:
df.display()

In [0]:

df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("catalog_name", "schema_name", "table_name") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_access_management.table_permissions;